In [ ]:
import torch
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from ml_translate.data import get_dataloader
from ml_translate.model import EncoderRNN, AttnDecoderRNN
from ml_translate.train import train
from ml_translate.eval import evaluate, evaluateRandomly

In [ ]:
hidden_size = 128
batch_size = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

input_lang, output_lang, pairs, train_dataloader = get_dataloader(batch_size, device)

encoder = EncoderRNN(input_lang.n_words, hidden_size).to(device)
decoder = AttnDecoderRNN(hidden_size, output_lang.n_words, device=device).to(device)

losses = train(train_dataloader, encoder, decoder, 80, print_every=5, plot_every=5)


def showPlot(points):
    plt.figure()
    fig, ax = plt.subplots()
    loc = ticker.MultipleLocator(base=0.2)
    ax.yaxis.set_major_locator(loc)
    plt.plot(points)


showPlot(losses)

Reading lines...
Read 135842 sentence pairs
Trimmed to 11445 sentence pairs
Counting words...
Counted words:
fra 4601
eng 2993
0 34.89337205886841 (- 8 43.40058088302612) (5 6.25) 1.5356555028334675
1 8.762163162231445 (- 8 1.3351421356201172) (10 12.5) 0.6962584585284388
1 46.43429207801819 (- 7 41.21526567141211) (15 18.75) 0.3630892314724416
2 26.04389214515686 (- 7 18.13167643547058) (20 25.0) 0.20212803449710653
2 58.49134087562561 (- 6 32.68094992637634) (25 31.25) 0.12469081136309565
3 34.32679581642151 (- 5 57.211326360702515) (30 37.5) 0.0858596602548434
4 8.154177904129028 (- 5 19.055371591023004) (35 43.75) 0.06424401503201968
4 58.75202298164368 (- 4 58.75202298164368) (40 50.0) 0.05251043419556577
5 34.58902406692505 (- 4 20.235907607608397) (45 56.25) 0.04536647573321558
6 9.635056972503662 (- 3 41.78103418350224) (50 62.5) 0.040466650149280296
6 42.404897928237915 (- 3 2.911317240108133) (55 68.75) 0.037199011114660084
7 15.449022054672241 (- 2 25.14967401822412) (60 75.

In [ ]:
encoder.eval()
decoder.eval()
evaluateRandomly(encoder, decoder, input_lang, output_lang, pairs, device)

> vous etes idiot
= you re being silly
< you are stupid stupid <EOS>

> je ne suis pas devin
= i m not a psychic
< i m not a psychic <EOS>

> il se plaint toujours de la nourriture
= he s always complaining about the food
< he s always complaining about the food <EOS>

> c est mon frere
= he is my brother
< he s my brother brother <EOS>

> ils sont inutiles
= they re useless
< i m useless <EOS>

> vous etes fiables
= you re reliable
< you re reliable <EOS>

> vous etes tres avisees
= you re very wise
< you re very wise <EOS>

> il va venir chez moi ce soir
= he is to come to my house tonight
< he is to come to my house tonight <EOS>

> je vous invite
= i m inviting you
< i m inviting you invited <EOS>

> elle est toujours libre l apres midi
= she is always free in the afternoon
< she is always free in the afternoon <EOS>



In [ ]:
def showAttention(input_sentence, output_words, attentions):
    fig = plt.figure()
    ax = fig.add_subplot(111)
    cax = ax.matshow(attentions.cpu().numpy(), cmap="bone")
    fig.colorbar(cax)

    # Show label at every tick
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    ax.yaxis.set_major_locator(ticker.MultipleLocator(1))

    # Set up axes
    ax.set_xticklabels([""] + input_sentence.split(" ") + ["<EOS>"], rotation=90)
    ax.set_yticklabels([""] + output_words)

    plt.show()


def evaluateAndShowAttention(input_sentence):
    output_words, attentions = evaluate(
        encoder, decoder, input_sentence, input_lang, output_lang, device
    )
    print("input =", input_sentence)
    print("output =", " ".join(output_words))
    showAttention(input_sentence, output_words, attentions[0, : len(output_words), :])


evaluateAndShowAttention("il n est pas aussi grand que son pere")
evaluateAndShowAttention("je suis trop fatigue pour conduire")
evaluateAndShowAttention("je suis desole si c est une question idiote")
evaluateAndShowAttention("je suis reellement fiere de vous")